# LPM_transplant — 03_host_foxf1_density_review

**Feeds:** Fig 5n

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


            # 03 | Host FOXF1 Density Review

            ## Notebook Scope

            This notebook visualizes the sampled pixel distributions as density maps and representative spatial plots for manuscript-facing interpretation.
            

In [ ]:
import os
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent.resolve()
else:
    ROOT = CWD

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.executable)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

In [ ]:
DENSITY_OUTPUT = ROOT / "results" / "tables" / "02_density_maps.tsv"
PIXEL_SAMPLE_OUTPUT = ROOT / "results" / "tables" / "02_sampled_pixel_profiles.tsv"
CLASS_SUMMARY_OUTPUT = ROOT / "results" / "tables" / "02_pixel_class_summary.tsv"

density_df = pd.read_csv(DENSITY_OUTPUT, sep="\t")
sampled_pixels_df = pd.read_csv(PIXEL_SAMPLE_OUTPUT, sep="\t")
class_summary_df = pd.read_csv(CLASS_SUMMARY_OUTPUT, sep="\t")

print("Density rows:", len(density_df))
print("Sampled pixels:", len(sampled_pixels_df))

In [ ]:
def plot_density_map(map_name: str, group_label: str) -> None:
    sub = density_df[
        (density_df["map_name"] == map_name)
        & (density_df["group_label"] == group_label)
        & (density_df["condition"] == "all_conditions")
    ].copy()
    if sub.empty:
        print(f"No rows for map={map_name!r}, group={group_label!r}, condition='all_conditions'")
        return

    x_bins = sorted(sub["x_bin_index"].unique().tolist())
    y_bins = sorted(sub["y_bin_index"].unique().tolist())
    grid = np.zeros((len(y_bins), len(x_bins)), dtype=np.float32)
    x_lookup = {v: i for i, v in enumerate(x_bins)}
    y_lookup = {v: i for i, v in enumerate(y_bins)}
    for row in sub.itertuples(index=False):
        grid[y_lookup[int(row.y_bin_index)], x_lookup[int(row.x_bin_index)]] = float(row.count)

    plt.figure(figsize=(6, 5))
    plt.imshow(np.log1p(grid), origin="lower", aspect="auto", cmap="magma")
    plt.colorbar(label="log1p(pixel count)")
    plt.title(f"{map_name} | {group_label} | all_conditions")
    plt.xlabel(sub["x_transform"].iloc[0])
    plt.ylabel(sub["y_transform"].iloc[0])
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_density_map_by_condition(map_name: str, group_label: str, condition: str) -> None:
    sub = density_df[
        (density_df["map_name"] == map_name)
        & (density_df["group_label"] == group_label)
        & (density_df["condition"] == condition)
    ].copy()
    if sub.empty:
        print(f"No rows for map={map_name!r}, group={group_label!r}, condition={condition!r}")
        return

    x_bins = sorted(sub["x_bin_index"].unique().tolist())
    y_bins = sorted(sub["y_bin_index"].unique().tolist())
    grid = np.zeros((len(y_bins), len(x_bins)), dtype=np.float32)
    x_lookup = {v: i for i, v in enumerate(x_bins)}
    y_lookup = {v: i for i, v in enumerate(y_bins)}
    for row in sub.itertuples(index=False):
        grid[y_lookup[int(row.y_bin_index)], x_lookup[int(row.x_bin_index)]] = float(row.count)

    plt.figure(figsize=(6, 5))
    plt.imshow(np.log1p(grid), origin="lower", aspect="auto", cmap="magma")
    plt.colorbar(label="log1p(pixel count)")
    plt.title(f"{map_name} | {group_label} | {condition}")
    plt.xlabel(sub["x_transform"].iloc[0])
    plt.ylabel(sub["y_transform"].iloc[0])
    plt.tight_layout()
    plt.show()

In [ ]:
for map_name, group_label in [
    ("host_vs_foxf1", "all_masked"),
    ("host_vs_foxf1", "host"),
    ("host_vs_foxf1", "donor"),
    ("donor_vs_host", "all_masked"),
]:
    plot_density_map(map_name, group_label)

In [ ]:
condition_list = [
    cond
    for cond in sorted(
        sampled_pixels_df["condition"].dropna().unique().tolist(),
        key=lambda cond: (0 if str(cond).lower() == "ctrl" else 1, str(cond).lower()),
    )
]

for condition in condition_list:
    plot_density_map_by_condition("host_vs_foxf1", "all_masked", condition)

In [ ]:
representative_planes = (
    class_summary_df[class_summary_df["pixel_class_label"] == "donor"]
    .sort_values(["fraction_of_mask", "foxf1_positive_fraction"], ascending=[False, False])
    .groupby("file_id", as_index=False)
    .head(1)
    [["condition", "file_id", "position_label", "file_name", "z_index"]]
    .sort_values(["condition", "file_id", "z_index"])
    .reset_index(drop=True)
)
display(representative_planes)

In [ ]:
CLASS_COLOR = {
    "unlabeled": "#9e9e9e",
    "host": "#2e7d32",
    "donor": "#1565c0",
    "mixed": "#ef6c00",
}


def plot_representative_spatial_samples(rep_df: pd.DataFrame) -> None:
    if rep_df.empty:
        print("No representative planes available.")
        return
    fig, axes = plt.subplots(len(rep_df), 2, figsize=(12, 5 * len(rep_df)))
    if len(rep_df) == 1:
        axes = np.asarray([axes])

    for ax_row, rep in zip(axes, rep_df.itertuples(index=False)):
        sub = sampled_pixels_df[
            (sampled_pixels_df["file_id"] == int(rep.file_id))
            & (sampled_pixels_df["z_index"] == int(rep.z_index))
        ].copy()
        if sub.empty:
            continue

        colors = sub["pixel_class_label"].map(CLASS_COLOR).fillna("#000000")
        ax_row[0].scatter(sub["col_px"], sub["row_px"], s=2, c=colors, alpha=0.65, linewidths=0)
        ax_row[0].invert_yaxis()
        ax_row[0].set_title(f"{rep.condition} | {rep.position_label} | z{rep.z_index}\nPixel classes")
        ax_row[0].set_aspect("equal")

        foxf1_host = sub[(sub["pixel_class_label"] == "host") & (sub["foxf1_positive"].astype(bool))].copy()
        base = sub[sub["pixel_class_label"] == "host"].copy()
        ax_row[1].scatter(base["col_px"], base["row_px"], s=2, c="#bdbdbd", alpha=0.25, linewidths=0)
        ax_row[1].scatter(foxf1_host["col_px"], foxf1_host["row_px"], s=4, c="#d32f2f", alpha=0.75, linewidths=0)
        ax_row[1].invert_yaxis()
        ax_row[1].set_title(
            f"{rep.condition} | {rep.position_label} | z{rep.z_index}\nHost pixels + FOXF1-positive host subset"
        )
        ax_row[1].set_aspect("equal")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_representative_spatial_samples(representative_planes)